# Fine-Tune RT-DETR — v2 Two-Stage Training

Fine-tune [RT-DETR r18vd](https://huggingface.co/PekingU/rtdetr_r18vd) on `kaya-go/moku-v2` using a two-stage approach:

1. **Stage 1 — Synthetic pre-training**: Learn general goban structure from ~1000 synthetic images (LR=1e-4, ~30 epochs)
2. **Stage 2 — Real fine-tuning**: Adapt to real-world domain on ~320 corrected real images (LR sweep: 1e-5 / 2e-5 / 5e-5, ~50 epochs)

Training runs on HF Jobs (A10G GPU). Results tracked via [W&B](https://wandb.ai/kaya-go/moku).

In [ ]:
%load_ext autoreload
%autoreload 2

from datasets import load_dataset
from transformers import Trainer, TrainingArguments

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.training import (
    collate_fn,
    evaluate_map,
    format_map_results,
    format_map_per_class,
    load_image_processor,
    load_model,
    make_eval_transform,
    make_train_transform,
    load_training_runs,
    summarize_runs,
)

HF_DATASET_V2 = "kaya-go/moku-v2"
HF_MODEL_V2 = "kaya-go/moku-v2"

## Load Datasets

Load both configs from `kaya-go/moku-v2`: synthetic (stage 1) and real (stage 2).

In [ ]:
synthetic_dataset = load_dataset(HF_DATASET_V2, "synthetic")
real_dataset = load_dataset(HF_DATASET_V2, "real")

print("Synthetic:", synthetic_dataset)
print("\nReal:", real_dataset)
print(f"\nCategories: {CATEGORIES}")
print(f"Labels: {ID_TO_CATEGORY}")

DatasetDict({
    train: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 382
    })
    validation: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 53
    })
    test: Dataset({
        features: ['image', 'image_id', 'width', 'height', 'source_dataset', 'objects'],
        num_rows: 50
    })
})

Categories: {'black_stone': 0, 'white_stone': 1, 'board_corner': 2}
Labels: {0: 'black_stone', 1: 'white_stone', 2: 'board_corner'}


## Load Model & Image Processor

Load RT-DETR r18vd pre-trained on COCO. Classification head re-initialized for 3 categories.

In [3]:
image_processor = load_image_processor()
model = load_model()

# Print trainable parameter count
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total:,}")
print(f"Trainable parameters: {trainable:,}")

Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r18vd
Key                                        | Status   |                                                                                        
-------------------------------------------+----------+----------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([3])          
model.enc_score_head.bias                  | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([3])          
model.denoising_class_embed.weight         | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([4, 256])
model.enc_score_head.weight                | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([3, 256])
model.decoder.class_embed.{0, 1, 2}.weight | MISMATCH | Reinit due to si

Total parameters: 20,075,740
Trainable parameters: 20,075,740


## Stage 1 — Synthetic Pre-training (local test)

Quick local test on CPU to verify the pipeline. For full training, use HF Jobs (see below).

In [ ]:
image_processor = load_image_processor()

# Apply transforms to synthetic data
synthetic_dataset["train"].set_transform(make_train_transform(image_processor))
synthetic_dataset["validation"].set_transform(make_eval_transform(image_processor))

# Verify a sample
sample = synthetic_dataset["train"][0]
print(f"pixel_values shape: {sample['pixel_values'].shape}")
print(f"labels keys: {sample['labels'].keys()}")
print(f"num objects: {len(sample['labels']['class_labels'])}")

pixel_values shape: torch.Size([3, 640, 640])
labels keys: KeysView({'size': tensor([640, 640]), 'image_id': tensor([0]), 'class_labels': tensor([2, 2, 2, 2]), 'boxes': tensor([[0.2164, 0.2188, 0.0266, 0.0344],
        [0.7641, 0.2266, 0.0250, 0.0312],
        [0.1477, 0.9789, 0.0422, 0.0422],
        [0.8250, 0.9809, 0.0344, 0.0367]]), 'area': tensor([374., 320., 729., 517.]), 'iscrowd': tensor([0, 0, 0, 0]), 'orig_size': tensor([640, 640])})
num objects: 4


In [ ]:
# Local test: 2 epochs on CPU just to verify pipeline
model = load_model()

training_args = TrainingArguments(
    output_dir="runs/v2_stage1_test",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    weight_decay=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    use_cpu=True,
    report_to="none",
    run_name="v2_stage1_test",
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=synthetic_dataset["train"],
    eval_dataset=synthetic_dataset["validation"],
)

trainer.train()
print("Stage 1 local test complete.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Output directory: runs/baseline
Epochs: 50
LR: 0.0001
Batch size: 4


## Launch Training on HF Jobs

Full training runs on HF Jobs with A10G GPU. Use `scripts/train.py` which supports:
- `--stage 1` for synthetic pre-training
- `--stage 2 --resume-from <model_id>` for real fine-tuning from a stage 1 checkpoint
- `--resume` to resume an interrupted run

### Stage 1: Synthetic Pre-training

```bash
hf jobs uv run \
    --detach --flavor a10g-small --timeout 3h --secrets HF_TOKEN \
    scripts/train.py \
    --stage 1 --run-name v2_stage1 --num-epochs 30
```

### Stage 2: LR Sweep (3 runs)

After stage 1 completes, launch 3 stage 2 runs with different LRs:

```bash
bash scripts/launch_grid.sh
```

Or manually:

```bash
for lr in 1e-5 2e-5 5e-5; do
    hf jobs uv run \
        --detach --flavor a10g-small --timeout 3h --secrets HF_TOKEN \
        scripts/train.py \
        --stage 2 --resume-from kaya-go/moku-v2-stage1 \
        --run-name "v2_stage2_lr${lr}" --lr $lr --num-epochs 50
done
```

In [ ]:
# Monitor running jobs
import subprocess
result = subprocess.run(["hf", "jobs", "ps"], capture_output=True, text=True)
print(result.stdout)

## Evaluate & Compare Stage 2 Runs

Load metrics from all stage 2 runs and select the best LR based on eval loss and mAP.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load all run histories
runs_dir = Path("runs")
if runs_dir.exists() and any(runs_dir.iterdir()):
    history_df = load_training_runs(runs_dir)
    summary_df = summarize_runs(runs_dir)

    print("=== Run Summary ===")
    display(summary_df.sort_values("eval_loss"))

    # Plot eval loss curves for stage 2 runs
    stage2_runs = history_df[history_df["run"].str.contains("stage2")]
    if not stage2_runs.empty:
        fig, ax = plt.subplots(figsize=(10, 5))
        for run_name, group in stage2_runs.groupby("run"):
            eval_data = group.dropna(subset=["eval_loss"])
            if not eval_data.empty:
                ax.plot(eval_data["epoch"], eval_data["eval_loss"], label=run_name, marker="o", markersize=3)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Eval Loss")
        ax.set_title("Stage 2 — Eval Loss by LR")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    # Find best run
    stage2_summary = summary_df[summary_df["run"].str.contains("stage2")]
    if not stage2_summary.empty:
        best_run = stage2_summary.loc[stage2_summary["eval_loss"].idxmin()]
        print(f"\n🏆 Best stage 2 run: {best_run['run']} (eval_loss={best_run['eval_loss']:.4f})")
else:
    print("No runs found yet. Launch training first.")

/Users/hadim/Code/libs/moku/.pixi/envs/default/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)


,metric,value
0,mAP,0.000060
1,mAP@50,0.000176
2,mAP@75,0.000000
3,mAR@100,0.007143


## Evaluate Best Model on Test Set

Load the best stage 2 model and compute COCO mAP on the real test split.

In [ ]:
# Load best model (update path after identifying best run)
BEST_RUN = "v2_stage2_lr2e-5"  # <-- update after LR sweep
best_model_path = f"runs/{BEST_RUN}"

best_model = load_model(model_name=best_model_path)
image_processor = load_image_processor()

# Evaluate on real test set
real_dataset["test"].set_transform(make_eval_transform(image_processor))

metrics = evaluate_map(
    model=best_model,
    dataset=real_dataset["test"],
    image_processor=image_processor,
    batch_size=8,
    threshold=0.3,
)

print("=== Overall mAP ===")
display(format_map_results(metrics))

print("\n=== Per-Class AP ===")
display(format_map_per_class(metrics))

## Push Best Model to Hub

Push the best stage 2 model as `kaya-go/moku-v2`.

In [ ]:
best_model.push_to_hub(HF_MODEL_V2)
image_processor.push_to_hub(HF_MODEL_V2)
print(f"Model pushed to https://huggingface.co/{HF_MODEL_V2}")